<a href="https://colab.research.google.com/github/muazzamasim04-ops/evolvix-aiml-internship-muazzamasim/blob/main/Week_2_Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# -------------------------------------------------------------------------
# 1. Load / Create Sample Data
# -------------------------------------------------------------------------
# Replace this block with your actual file loading code:
# df = pd.read_csv('your_dataset.csv')

np.random.seed(42)
n_samples = 500
dates = pd.date_range(start="2023-01-01", periods=n_samples, freq="D")
df = pd.DataFrame({
    'date': dates,
    'feature_A': np.random.uniform(10, 100, size=n_samples),
    'feature_B': np.random.uniform(1, 10, size=n_samples),
    'numeric_col': np.random.normal(50, 15, size=n_samples)
})
# Target formula for demonstration
df['target'] = (
    df['feature_A'] * 1.5
    + df['feature_B'] * 3.0
    + df['date'].dt.dayofweek * 2.5
    + np.random.normal(0, 5, size=n_samples)
)

# -------------------------------------------------------------------------
# 2. Feature Engineering (At least 2 new features)
# -------------------------------------------------------------------------
# Feature 1: Date Component Extraction
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek

# Feature 2: Ratio / Interaction Feature
df['feature_ratio'] = df['feature_A'] / (df['feature_B'] + 1e-5)

# Feature 3: Binning a Numeric Feature
df['numeric_binned'] = pd.qcut(df['numeric_col'], q=4, labels=[0, 1, 2, 3]).astype(int)

# -------------------------------------------------------------------------
# 3. Data Preparation
# -------------------------------------------------------------------------
# Drop non-numeric original date column and target
X = df.drop(columns=['target', 'date'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------------------------------------------------------------------------
# 4. Baseline Model Evaluation
# -------------------------------------------------------------------------
baseline_model = XGBRegressor(random_state=42)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)
baseline_score = np.sqrt(mean_squared_error(y_test, baseline_preds))

# -------------------------------------------------------------------------
# 5. Hyperparameter Tuning using GridSearchCV
# -------------------------------------------------------------------------
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
tuned_preds = best_model.predict(X_test)
tuned_score = np.sqrt(mean_squared_error(y_test, tuned_preds))

# -------------------------------------------------------------------------
# 6. Performance Comparison
# -------------------------------------------------------------------------
improvement = ((baseline_score - tuned_score) / baseline_score) * 100

print("\n--- Performance Summary ---")
print(f"Best Hyperparameters : {grid_search.best_params_}")
print(f"Baseline RMSE        : {baseline_score:.4f}")
print(f"Tuned Model RMSE     : {tuned_score:.4f}")
print(f"Quantified Improvement: {improvement:.2f}% reduction in error")

Fitting 5 folds for each of 24 candidates, totalling 120 fits

--- Performance Summary ---
Best Hyperparameters : {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Baseline RMSE        : 7.2378
Tuned Model RMSE     : 5.8710
Quantified Improvement: 18.88% reduction in error
